## Algoritmo método de Newton
```
Paso 1  Tome i = 1.
Paso 2  Mientras i <= N0 haga los pasos 3 a 6.
Paso 3      Tome p = p0 - f(p0) / f'(p0).
Paso 4      Si |p - p0| < TOL: SALIDA (p); PARE.
Paso 5      Tome i = i + 1.
Paso 6      Tome p0 = p.
Paso 7  SALIDA ("El método fracasó después de N0 iteraciones"); PARE.

```

In [1]:
def newton_raphson(f, df, p0, TOL=1e-6, N0=50):
    """Algoritmo 2.3. Devuelve (p, historia), con historia = [p0, p1, ..., p]."""
    historia = [p0]
    i = 1
    while i <= N0:
        d = df(p0)
        if d == 0:
            raise ZeroDivisionError(f"f'(p0) = 0 en p0 = {p0}: la tangente es horizontal")
        p = p0 - f(p0) / d                          # COMPLETE: Paso 3, usando d
        historia.append(p)             # COMPLETE
        if abs(p - p0) < TOL:                          # COMPLETE: Paso 4
            return p, historia
        i = i + 1
        p0 = p                         # COMPLETE: Paso 6
    raise RuntimeError(f"El método fracasó después de {N0} iteraciones")

¿Qué solución p devuelve su función?

In [2]:
def f(x):
    return x**3 +4*x**2 - 10

def df(x):
    return 3*x**2 + 8*x

p, historia = newton_raphson(f, df, 1.5, TOL=1e-6)
print(p)


1.3652300134140969


Escriba $p_1, p_2, p_3$ de su historia

In [3]:
print(historia)

[1.5, 1.3733333333333333, 1.3652620148746266, 1.3652300139161466, 1.3652300134140969]


## Algoritmo método de secante
```text
Paso 1  Tome i = 2; q0 = f(p0); q1 = f(p1). 
Paso 2  Mientras i <= N0 haga los pasos 3 a 6. 
Paso 3      Tome p = p1 - q1 (p1 - p0) / (q1 - q0). 
Paso 4      Si |p - p1| < TOL: SALIDA (p); PARE. 
Paso 5      Tome i = i + 1. 
Paso 6      Tome p0 = p1; q0 = q1; p1 = p; q1 = f(p). 
Paso 7  SALIDA ("El método fracasó después de N0 iteraciones"); PARE.
```

In [4]:
def secante(f, p0, p1, TOL=1e-6, N0=50):
    """Algoritmo 2.4. Devuelve (p, historia), con historia = [p0, p1, p2, ..., p]."""
    # Paso 1: i = 2; q0 = f(p0); q1 = f(p1); historia empieza con p0 y p1.
    i = 2
    q0 = f(p0)
    q1 = f(p1)
    historia = [p0, p1]
    # Paso 2: mientras i <= N0, haga los pasos 3 a 6.
    while i <= N0:
        # Si q1 == q0, la secante es horizontal: lance un ZeroDivisionError con un mensaje claro.
        if q1 == q0:
            raise ZeroDivisionError("La secante es horizontal: f(p1) == f(p0).")
        # Paso 3: calcule p y agréguelo a la historia.
        p = p1 - q1 * (p1 - p0) / (q1 - q0)
        historia.append(p)
        # Paso 4: si |p - p1| < TOL, devuelva (p, historia).
        if abs(p - p1) < TOL:
            return p, historia
        # Paso 5: i = i + 1.
        i += 1
        # Paso 6: corra los puntos (p0, q0) <- (p1, q1) y (p1, q1) <- (p, f(p)).
        #         Es la única evaluación nueva de f en la iteración.
        p0, q0 = p1, q1
        p1, q1 = p, f(p)

    # Paso 7: lance un RuntimeError: "El método fracasó después de N0 iteraciones".
    raise RuntimeError(f"El método fracasó después de {N0} iteraciones.")


Cuente las evaluaciones de f. En clase pusimos un print dentro de f para ver cada evaluación. Ahora las guardamos en una lista:

In [5]:
llamadas = []            # cada evaluación de f agrega aquí su x

def f(x):
    llamadas.append(x)
    return x**3 + 4*x**2 - 10

llamadas.clear()         # vacíe la lista antes de cada prueba
p, historia = secante(f, 1.0, 2.0, TOL=1e-6)
print(p, len(historia) - 2, len(llamadas))

1.3652300134142061 6 7


Use si función con $f(x) = x^3 +4x^2 -10, p_0 = 2, p_1 = 2$  y  $TOL = 10^{-6}$

In [6]:
def f(x):
    return x**3 +4*x**2 - 10

p, historia = secante(f, 1.0, 2.0, TOL=1e-6)
print(p,historia)
print(len(llamadas))

1.3652300134142061 [1.0, 2.0, 1.263157894736842, 1.3388278388278387, 1.3666163947193453, 1.3652119026318565, 1.3652300011108591, 1.3652300134142061]
7


### Ejerccio 3. ¿Qué tasa de interés cobra el banco?
Un préstamo de $20000 para un auto se paga en 60 cuotas mensuales de $450. Cada mes, el saldo crece con el interés mensual i y baja con la cuota:

$$ 
s_k = s_{k-1}(1+i)-450,\quad s_0 = 20000 
$$
La tasa del préstamo es la que deja el saldo en cero después de la cuota 60: buscamos la raíz de $S(i) = s_{00}$.

```python
def saldo_final(i, monto=20000.0, cuota=450.0, meses=60):
    """Saldo después de pagar todas las cuotas, con interés mensual i (0.01 es 1 %)."""
    saldo = monto
    for mes in range(meses):
        saldo = ...            # COMPLETE: interés del mes y pago de la cuota
    return saldo

```
saldo_final es una *caja negra*: un un ciclo, no una fórmula. No tenemos su derivada, así que usamos la secante.

1. Complete la funcion y pruébela con un caso que pueda calcular a mano: sin interés.
2. Ecuentre $i$ con secante(saldo_final, 0.005, 0.02, TOL=1e-10).
3. Calcule la tasa efectiva anual, $(1+i)^{12}-1$



In [7]:
def saldo_final(i, monto=20000.0, cuota=450.0, meses=60):
    """Saldo después de pagar todas las cuotas, con interés mensual i (0.01 es 1 %)."""
    saldo = monto
    for mes in range(meses):
        saldo = saldo * (1 + i) - cuota            # COMPLETE: interés del mes y pago de la cuota
    return saldo

¿Cuánto vale saldo_final(0)?

In [8]:
print(saldo_final(0))

-7000.0


¿Cuál es la tasa de interés mensual i, en porcentaje? Escríbala con 4 decimales (por ejemplo, 0.0123 se escribe 1.2300).

In [9]:
i, historia = secante(saldo_final, 0.005, 0.02, TOL=1e-10)
print("Tasa mensual:", i)
print(f"Tasa mensual en porcentaje: {i*100:.4f}%")

Tasa mensual: 0.010420044582509077
Tasa mensual en porcentaje: 1.0420%


¿Cuál es la tasa efectiva anual, en porcentaje? Escríbala con 2 decimales.

In [10]:
tasa_anual = (1 + i)**12 - 1
print(f"Tasa anual:", tasa_anual)
print(f"Tasa anual en porcentaje: {tasa_anual*100:.4f}%")

Tasa anual: 0.1324614765024601
Tasa anual en porcentaje: 13.2461%
